У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier

In [2]:
df = pd.read_csv('/content/customer_segmentation_train.csv')

In [3]:
df.head()

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [5]:
df = df.drop('ID', axis=1)
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['Segmentation'])

target_col = 'Segmentation'

train_inputs = train_df.drop(columns=[target_col])
train_targets = train_df[target_col]
test_inputs = test_df.drop(columns=[target_col])
test_target = test_df[target_col]

numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes(include=['object']).columns.tolist()

In [12]:
# конвеєр для числових даних
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# конвеєр для категоріальних даних
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
])

# Комбінуємо трансформери для різних типів колонок в один препроцесор
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [13]:
train_processed = preprocessor.fit_transform(train_inputs)
test_processed = preprocessor.transform(test_inputs)

In [14]:
new_column_names = preprocessor.get_feature_names_out()
train_processed_df = pd.DataFrame(train_processed, columns=new_column_names)
print(train_processed_df.head())

  num__Age num__Work_Experience num__Family_Size cat__Gender  \
0     32.0                  9.0              1.0      Female   
1     72.0                  1.0              2.0        Male   
2     33.0                  1.0              4.0      Female   
3     48.0                  0.0              6.0      Female   
4     28.0                  9.0              1.0      Female   

  cat__Ever_Married cat__Graduated cat__Profession cat__Spending_Score  \
0                No            Yes          Artist                 Low   
1               Yes            Yes   Entertainment             Average   
2                No            Yes   Entertainment                 Low   
3               Yes            Yes          Artist             Average   
4               Yes             No          Doctor                 Low   

  cat__Var_1  
0      Cat_6  
1      Cat_6  
2      Cat_6  
3      Cat_6  
4      Cat_7  


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [21]:
from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.combine import SMOTETomek

train_num_only = train_processed[:, :len(numeric_cols)]
base_smote = SMOTE(random_state=42)
train_num_smote, train_target_smote = base_smote.fit_resample(train_num_only, train_targets)

cat_indx = list(range(len(numeric_cols), train_processed.shape[1]))
base_smotenc = SMOTENC(categorical_features=cat_indx, random_state=42)
train_mixed_smotenc, train_target_smotenc = base_smotenc.fit_resample(train_processed, train_targets)

base_smotet = SMOTETomek(smote=base_smotenc, random_state=42)
train_mixed_smotet, train_target_smotet = base_smotenc.fit_resample(train_processed, train_targets)

In [22]:
train_processed.shape, train_mixed_smotenc.shape, train_mixed_smotet.shape

((6454, 9), (7256, 9), (7256, 9))

In [26]:
classes, counts = np.unique(train_target_smotenc, return_counts=True)
print(train_targets.value_counts())
print(dict(zip(classes, counts)))

Segmentation
D    1814
A    1578
C    1576
B    1486
Name: count, dtype: int64
{'A': np.int64(1814), 'B': np.int64(1814), 'C': np.int64(1814), 'D': np.int64(1814)}


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [31]:
final_num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

final_cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

final_preprocessor = ColumnTransformer(
    transformers=[
        ('num', final_num_transformer, numeric_cols),
        ('cat', final_cat_transformer, categorical_cols)
])

new_cols = numeric_cols + categorical_cols

train_orig_final = final_preprocessor.fit_transform(train_inputs)
train_smote_final = final_preprocessor.transform(pd.DataFrame(train_mixed_smotenc, columns=new_cols))
train_tomek_final = final_preprocessor.transform(pd.DataFrame(train_mixed_smotet, columns=new_cols))

test_final = final_preprocessor.transform(test_inputs)

In [34]:
# Логістична регресія зі стратегією one-vs-rest (OvR)
# Створюю три окремі моделі
model_orig = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
model_smote = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
model_tomek = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))

# Оригінальні дані
model_orig.fit(train_orig_final, train_targets)
y_pred_orig = model_orig.predict(test_final)
print('Оригінальні дані')
print(classification_report(test_target, y_pred_orig))

# SMOTENC
model_smote.fit(train_smote_final, train_target_smotenc)
y_pred_smote = model_smote.predict(test_final)
print('SMOTENC')
print(classification_report(test_target, y_pred_smote))

# SMOTE-TOMEK
model_tomek.fit(train_tomek_final, train_target_smotet)
y_pred_tomek = model_tomek.predict(test_final)
print('SMOTE-TOMEK')
print(classification_report(test_target, y_pred_tomek))

Оригінальні дані
              precision    recall  f1-score   support

           A       0.41      0.45      0.43       394
           B       0.40      0.17      0.24       372
           C       0.49      0.61      0.54       394
           D       0.64      0.76      0.70       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.48      1614
weighted avg       0.49      0.51      0.49      1614

SMOTENC
              precision    recall  f1-score   support

           A       0.42      0.47      0.44       394
           B       0.41      0.26      0.32       372
           C       0.52      0.60      0.55       394
           D       0.67      0.72      0.69       454

    accuracy                           0.52      1614
   macro avg       0.51      0.51      0.50      1614
weighted avg       0.51      0.52      0.51      1614

SMOTE-TOMEK
              precision    recall  f1-score   support

           A       0.42      0.47     

Для порівняння оберу macro(F1-score) - метрика вираховує якість для кожного класу окремо, а потім бере середнє арифметичне.  
Найкращий результат показали моделі на збалансованих даних - SMOTENC та SMOTE-TOMEK. Вони мають абсолютно однакові метрики, які кращі за оригінальну модель.  
Однаковость методів (SMOTENC та SMOTE-TOMEK) можливо обусловлена слабким початковим дисбалансом даних. Або можливо клієнти різних сегментів дуже сильно "перемішані" між собою за ознаками і логистичної регрессії важко провести між ними лінії-межі.